# 03 — LightGBM benchmark

A high-performance gradient-boosted tree benchmark. It uses shared features, native numeric missing-value handling, training-only frequency encoding of categoricals, and class weighting.

In [ ]:
!pip -q install lightgbm pandas pyarrow scikit-learn
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, early_stopping
from sklearn.metrics import roc_auc_score, average_precision_score

DATA = Path('/content/drive/MyDrive/ieee_fraud/processed')
ARTIFACTS = Path('/content/drive/MyDrive/ieee_fraud/artifacts'); ARTIFACTS.mkdir(exist_ok=True)
train, valid, test = [pd.read_parquet(DATA / f'{x}.parquet') for x in ['train', 'validation', 'test']]
features = json.loads((DATA / 'preparation_metadata.json').read_text())['feature_columns']
X_train, X_valid = train[features].copy(), valid[features].copy()
y_train, y_valid = train.isFraud, valid.isFraud
categorical = X_train.select_dtypes(exclude=np.number).columns.tolist()

# Fit category frequency maps on training rows only.
for col in categorical:
    mapping = X_train[col].fillna('MISSING').astype(str).value_counts(normalize=True)
    X_train[col] = X_train[col].fillna('MISSING').astype(str).map(mapping).fillna(0).astype('float32')
    X_valid[col] = X_valid[col].fillna('MISSING').astype(str).map(mapping).fillna(0).astype('float32')

weight = (y_train == 0).sum() / (y_train == 1).sum()
model = LGBMClassifier(n_estimators=3000, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=weight, random_state=42, n_jobs=-1)
started = time.time()
model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], eval_metric='auc', callbacks=[early_stopping(100, verbose=100)])
probability = model.predict_proba(X_valid)[:, 1]
metrics = {'model': 'lightgbm', 'roc_auc': float(roc_auc_score(y_valid, probability)), 'pr_auc': float(average_precision_score(y_valid, probability)), 'training_seconds': round(time.time() - started, 2), 'best_iteration': int(model.best_iteration_)}
print(metrics)
(ARTIFACTS / 'lightgbm_metrics.json').write_text(json.dumps(metrics, indent=2))
model.booster_.save_model(str(ARTIFACTS / 'lightgbm_v1.txt'))